In [1]:
# Set up a User Agent for this session
import os

os.environ['USER_AGENT'] = 'sports-buddy-advanced'

In [2]:
# Initialize an OpenAI model
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WikipediaLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WikipediaLoader("2024_Summer_Olympics",)
docs = loader.load()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
splits = text_splitter.split_documents(docs)

database = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

# TODO: Increase the value of 'k' to retrieve more documents
retriever = database.as_retriever(search_kwargs={"k": 1})

response = retriever.invoke("How was security during the 2024 Olympics")
print(len(response))

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    "How was security during the 2024 Olympics"
)

compressed_docs

In [ ]:
pip install langchain-cohere

In [ ]:
import os
import getpass

if not os.getenv("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Enter your Cohere API Key:")

In [ ]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank
from langchain_community.llms import Cohere

llm = Cohere(temperature=0)
compressor = CohereRerank(model="rerank-english-v3.0")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    "How was security during the 2024 Olympics"
)

compressed_docs